<a href="https://colab.research.google.com/github/Kunalpatel-08/Bank-Marketing-Effectiveness-Prediction/blob/main/kunal_bank_propensity.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Project Name**    -



##### **Project Type**    - Classification ML

##### **Contribution**    - Individual

# **Project Summary -**

Write the summary here within 500-600 words.

# **GitHub Link -**

Provide your GitHub Link here.

# **Problem Statement**


The data is related to direct marketing campaigns (phone calls) of a Portuguese banking institution. The marketing campaigns were based on phone calls. Often, more than one contact to the same client was required, in order to assess if the product (bank term deposit) would be ('yes') or not ('no') subscribed. The classification goal is to predict if the client will subscribe to a term deposit (variable y).



# **General Guidelines** : -  

1.   Well-structured, formatted, and commented code is required.
2.   Exception Handling, Production Grade Code & Deployment Ready Code will be a plus. Those students will be awarded some additional credits.
     
     The additional credits will have advantages over other students during Star Student selection.
       
             [ Note: - Deployment Ready Code is defined as, the whole .ipynb notebook should be executable in one go
                       without a single error logged. ]

3.   Each and every logic should have proper comments.
4. You may add as many number of charts you want. Make Sure for each and every chart the following format should be answered.
        

```
# Chart visualization code
```
            

*   Why did you pick the specific chart?
*   What is/are the insight(s) found from the chart?
* Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

5. You have to create at least 15 logical & meaningful charts having important insights.


[ Hints : - Do the Vizualization in  a structured way while following "UBM" Rule.

U - Univariate Analysis,

B - Bivariate Analysis (Numerical - Categorical, Numerical - Numerical, Categorical - Categorical)

M - Multivariate Analysis
 ]





6. You may add more ml algorithms for model creation. Make sure for each and every algorithm, the following format should be answered.


*   Explain the ML Model used and it's performance using Evaluation metric Score Chart.


*   Cross- Validation & Hyperparameter Tuning

*   Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

*   Explain each evaluation metric's indication towards business and the business impact pf the ML model used.




















# ***Let's Begin !***

## ***1. Know Your Data***

### Import Libraries

In [ ]:
# Import Library

import os, json, time
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

from scipy import stats

from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler,FunctionTransformer
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import (train_test_split, StratifiedKFold, GridSearchCV, RandomizedSearchCV, cross_val_predict, cross_val_score)
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
                             average_precision_score, brier_score_loss, log_loss, roc_curve,
                             precision_recall_curve, confusion_matrix)
from sklearn.calibration import calibration_curve
from sklearn.inspection import permutation_importance
import joblib
from xgboost import XGBClassifier


# ---- Global Settings -------------------------------------------------------------------------------------------------

pd.set_option("display.max_columns",50); pd.set_option("display.width",200)
pd.set_option("display.float_format",lambda v: f"{v:,.4f}")
sns.set_theme(style="whitegrid", context="notebook")

plt.rcParams.update({"figure.dpi":100,
                     "axes.titleweight":"bold",
                     "axes.titlesize":12,
                     "axes.spines.top":False,
                     "axes.spines.right": False
                     })

COL_NO, COL_YES = "#7A8CA5", "#E4572E"  # consistent colours: non-subscriber / subscriber

RANDOM_STATE = 42
FAST_MODE = os.environ.get("FAST_MODE") == "1"    # True => tiny hyper-parameter searches (smoke test only)
np.random.seed(RANDOM_STATE)
print("Libraries Loaded | FAST_MODE=", FAST_MODE)



### Dataset Loading

In [ ]:
# Load Dataset

# Load the Dataset from GitHub Repo :

# 1. Store GitHub raw dataset URL
DATASET_URL = "https://raw.githubusercontent.com/Kunalpatel-08/Bank-Marketing-Effectiveness-Prediction/refs/heads/main/bank-full.csv"

# 2. Define a local filename fallback
LOCAL_FILE = "bank-full.csv"

# 3. Load dataset with semicolon separator
try:
    print("Loading dataset from GitHub...")
    raw = pd.read_csv(DATASET_URL, sep=';')   # # sep=';' tells pandas to split columns by semicolons instead of commas
    print("Dataset loaded successfully from GitHub!")
except Exception as e:
    print(f"Error loading from GitHub: {e}")
    if os.path.exists(LOCAL_FILE):
        print("Loading from local fallback...")
        raw = pd.read_csv(LOCAL_FILE, sep=';')

### Dataset First View

In [ ]:
# Dataset First Look
raw.head(10)

### Dataset Rows & Columns count

In [ ]:
# Dataset Rows & Columns count
print(f"Rows    : {raw.shape[0]:,}")
print(f"Columns : {raw.shape[1]}")

### Dataset Information

In [ ]:
# Dataset Info
raw.info()

#### Duplicate Values

In [ ]:
# Dataset Duplicate Value Count
print(f"Fully Duplicated Rows :", raw.duplicated().sum())

#### Missing Values/Null Values

In [ ]:
# Missing Values/Null Values Count
nulls = raw.isnull().sum()
print(nulls.to_string())
print("\n Total Nulls cells :",int(nulls.sum()))

In [ ]:
# Target Variable Distibution
class_counts = raw["y"].value_counts()
class_pct = raw["y"].value_counts(normalize=True) * 100

majority = class_counts.max()
minority = class_counts.min()

imbalance_ratio = majority / minority

print("Target Variable Distribution")
print("-" * 35)

for cls in class_counts.index:
    print(f"{cls}: {class_counts[cls]:,} ({class_pct[cls]:.2f}%)")

print(f"\nImbalance Ratio: {imbalance_ratio:.2f}:1")

In [ ]:
# Visualizing the missing values
# The dataset have no Null Values but missingness in the dataset is often represented as " unknown" values in the respective columns
# & pdays uses sentinel as -1 (= client that have never been contacted before). Both are "hidden Missing" values.

cat_cols_raw = raw.select_dtypes(exclude="number").columns.drop("y")
hidden = raw[cat_cols_raw].eq("unknown").mean()*100
hidden["pdays  (-1 = never contacted)"] = raw["pdays"].eq(-1).mean()*100
hidden = hidden[hidden > 0].sort_values()

# Ploth the graph
fig, ax = plt.subplots(1, 2, figsize=(13,4.5), gridspec_kw={"width_ratios": [1, 1.4]})
nulls.sort_values().plot.barh(ax=ax[0], color=COL_NO)
ax[0].set(title="Formal Null Count per COLUMN", xlabel="Null cells"); ax[0].set_xlim(0, 1)
ax[0].text(0.5, 8, "0 Nulls in 17 cloumns", ha="center", fontsize=12, fontweight="bold", color="green")
hidden.plot.barh(ax=ax[1], color=COL_YES)
for i, v in enumerate(hidden.values):
  ax[1].text(v+1, i, f"{v:,.1f}", va="center", fontweight="bold")
ax[1].set(title="Hidden missing values ('unknown'/ sentinel)", xlabel="% of rows"); ax[1].set_xlim(0, 100)
plt.tight_layout()
plt.show()

### What did you know about your dataset?

- **45,211 rows, 17 columns, target `y`** (yes/no, 11.70% positive, i.e ~7.5:1 class imbalance).
- **No formal NULLs and no duplicate rows** - but missingness is *hidden* behind sentinel values: `poutcome`
  and `pdays` are "unknown"/`-1` for 81.7% of clients (never contacted in a previous campaign), `contact` is
  "unknown" for 28.8%, and `job`/`education` have small "unknown" shares too.
- **`duration` is a post-outcome field** (i.e call length is only known once the call has ended, so it cannot be used
  by a model which has to make decision that *whether to call in the first place*) - flagged immediately and handled explicitly
  throughout rather than silently included.

## ***2. Understanding Your Variables***

In [ ]:
# Dataset Columns
print(list(raw.columns))

In [ ]:
# Dataset Describe
display(raw.describe().T)
display(raw.describe(exclude="number").T)

### Variables Description

| Column | Type | Meaning |
|---|---|---|
| age | numeric | client's age in years |
| job | categorical | occupation (12 levels) |
| marital | categorical | marital status |
| education | categorical | education level |
| default | categorical | has credit in default? |
| balance | numeric | average yearly account balance (EUR) |
| housing | categorical | has a housing loan? |
| loan | categorical | has a personal loan? |
| contact | categorical | contact communication type (cellular/telephone/unknown) |
| day | numeric | last contact day of the month |
| month | categorical | last contact month of year |
| duration | numeric | last contact duration, in seconds - **post-call, not usable pre-call** |
| campaign | numeric | number of contacts performed during this campaign for this client |
| pdays | numeric | days since last contact from a previous campaign; -1 = never contacted |
| previous | numeric | number of contacts before this campaign |
| poutcome | categorical | outcome of the previous marketing campaign |
| y | binary target | did the client subscribe to a term deposit? |

### Check Unique Values for each variable.

In [ ]:
# Check Unique Values for each variable.

uniq = pd.DataFrame({
    "dtype": raw.dtypes.astype(str),
    "n_unique": raw.nunique(),
    "example_values":[ raw[c].drop_duplicates().head(6).tolist() for c in raw.columns]
})

## 3. ***Data Wrangling***

### Data Wrangling Code

In [ ]:
raw.sample(2)

In [ ]:
# Write your code to make your dataset analysis ready.

df = raw.copy()

# 1) Target ---> 0/1
df["y"] = (df[["y"]] == "yes").astype(int)

# 2) pdays sentinels : -1 means "not previously contacted".
    #  keeping the information in two honest features : a NAN-able "days since last contact" and an explicit binary flag.

df["pdays_clean"]    = df["pdays"].replace(-1, np.nan)
df["prev_contacted"] = df["pdays"].ne(-1).astype(int)


# 3) Engineered Features (all derivable before the CALL is made)

df["n_loans"]          = ((df["housing"] == "yes").astype(int) + (df["loan"] == "yes").astype(int)).astype(str)  # 0/1/2 debts
df["balance_negative"] = (df["balance"] < 0).astype(int)
df["balance_slog"]     = np.sign(df["balance"]) * np.log1p(df["balance"].abs())     # For Exploration only
df["campaign_capped"]  = df["campaign"].clip(upper=df["campaign"].quantile(0.99))   # For Exploration only

df["age_group"]        = pd.cut(df["age"], [17, 24, 34, 44, 54, 64, 120],
                                labels=["<25", "25-34", "35-44", "45-54", "55-64", "65+"])

# Check for Chronology . The UCI dataset is ordered by date (May 2008 -> Nov 2010). Let's verify instead of assuming:
df["row_id"]  = np.arange(len(df))
month_blocks = (df["month"] != df["month"].shift()).cumsum()
print(
    f"Reading top-to-bottom the 'month' column forms "
    f"{month_blocks.nunique()} contiguous calendar-month blocks"
    f"(first: {df['month'].iloc[0]}, last: {df['month'].iloc[-1]})"
    f"-> rows are in chronological orders"
)

# 'period' is used ONLY for the stability (PSI) analysis - never as a model feature.

df["period"]  = np.where(df["row_id"] < 0.7 * len(df), "early (first 70% of rows)", "late (last 30% of rows)")

MONTH_ORDER   = ["jan", "feb", "mar", "apr", "may", "jun", "jul", "aug", "sep", "oct", "nov", "dec"]

BASE_RATE = df["y"].mean()

def rate_table(data, col, order=None):
    """Customers / subscribers / subscription-rate per level of `col`."""
    t = data.groupby(col, observed=True)["y"].agg(customers="size", subscribers="sum", rate="mean")
    return t.reindex(order) if order is not None else t

# 5) Model inputs.

CAT_FEATURES = ["job", "marital", "education", "default", "n_loans", "contact", "month", "poutcome"]
NUM_FEATURES = ["age", "balance", "day", "campaign", "pdays_clean"]
LEAKY        = ["duration"]

EDA_EXTRA    = ["housing", "loan", "previous"]

print(f"\nBase subscription rate: {BASE_RATE:.2%}   |   analysis-ready shape: {df.shape}")
df[["age","job","balance","pdays","pdays_clean","prev_contacted","n_loans","balance_negative","age_group","y"]].head()



### What all manipulations have you done and insights you found?

- Target mapped to 0/1. `pdays`'s -1 sentinel is split into an honest `pdays_clean` (NaN when never contacted)
  and a binary `prev_contacted` flag, since -1 is not really "zero days" - it is "not applicable".
- Engineered `n_loans` (0/1/2 = count of housing + personal loans), `balance_negative`, a signed-log transform of
  balance for skew diagnostics, a winsorised campaign count for plotting, and an `age_group` band.
- Chronological order confirmed and stored as `period` (`early` = first 70% of rows, `late` = last 30%) - used
  **only** in the PSI section, never as a model input.
- The deployable feature set (`FEATS`) deliberately excludes `duration` (leakage), and - after the checks in
  Section 6 - also excludes `housing`, `loan` and `previous` (redundant with `n_loans` and unstable under WOE);
  those three are kept in the working frame for exploration only.

## ***4. Data Vizualization, Storytelling & Experimenting with charts : Understand the relationships between variables***

#### Chart - 1 - Target Distribution ( Univariate )

In [ ]:
# Chart - 1 visualization code

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
counts = df["y"].map({0: "No", 1: "Yes"}).value_counts()
sns.barplot(x=counts.index, y=counts.values, hue=counts.index, palette={"No": COL_NO, "Yes": COL_YES}, legend=False, ax=ax[0])
for i, v in enumerate(counts.values):
    ax[0].text(i, v + 500, f"{v:,}\n({v/len(df):.1%})", ha="center", fontweight="bold")
ax[0].set(title="Term-deposit subscription (target y)", xlabel="Subscribed?", ylabel="Clients"); ax[0].set_ylim(0, counts.max() * 1.18)
ax[1].pie(counts.values, labels=counts.index, autopct="%1.1f%%", startangle=90, colors=[COL_NO, COL_YES],
          wedgeprops=dict(width=0.20, edgecolor="white"))
ax[1].set_title("Share of outcomes")
plt.tight_layout(); plt.show()
print(f"Imbalance ratio  No : Yes = {counts['No']/counts['Yes']:.1f} : 1")

##### 1. Why did you pick the specific chart?

A bar + donut pie chart combo is the simplest, least ambiguous way to show a binary target's split and its exact imbalance ratio.

##### 2. What is/are the insight(s) found from the chart?

Only 11.7% of contacted clients subscribed - a 7.5:1 imbalance. Accuracy alone would be a misleading metric here (predicting 'no' for everyone already scores 88.3%).


##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Positive: confirms the campaign is inherently low-hit-rate, which is exactly why a propensity model with cost-aware targeting.


#### Chart - 2 - Age Distribution ( Univariate )

In [ ]:
# Chart - 2 visualization code

fig, ax = plt.subplots(1,2, figsize=(12,4), gridspec_kw={"width_ratios": [2.2,1]})
sns.histplot(df["age"], bins=40, kde=True, color=COL_NO, ax=ax[0])
ax[0].axvline(df["age"].median(), color=COL_YES, ls="--", label=f"median = {df["age"].median():.0f}")
ax[0].set(title="Distribution of client age", xlabel="Age (years)", ylabel="Clients")
ax[0].legend()

sns.boxplot(y=df["age"], color=COL_NO, ax=ax[1])

ax[1].set(title="Boxplot", ylabel="Age")
plt.tight_layout()
plt.show()
print("\n")
print(df["age"].describe().round(1).to_string())

##### 1. Why did you pick the specific chart?

A histogram+KDE shows the overall shape of the age distribution, and a boxplot alongside makes outliers/skew easy to spot at a glance.

##### 2. What is/are the insight(s) found from the chart?

Age is right-skewed and concentrated between 30-50 (median 39); a long tail extends to 95. No impossible values (min 18) were found.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Answer Here

#### Chart - 3 - Job Profile of Contacted Client ( Univariate )

In [ ]:
# Chart - 3 visualization code

job_counts = df["job"].value_counts()

fig, ax = plt.subplots(figsize=(9, 4.6))
sns.barplot(x=job_counts.values, y=job_counts.index, hue=job_counts.index, palette="mako", legend=False, ax=ax)
for i,v in enumerate(job_counts.values):
    ax.text(v + 80, i, f"{v:,} ({v/len(df):.1%})", va="center", fontsize=9)

ax.set(title="Clients contacted by occupation", xlabel="clients", ylabel="")
ax.set_xlim(0, job_counts.max() * 1.22)
plt.tight_layout()
plt.show()


##### 1. Why did you pick the specific chart?

A horizontal bar chart ranks all 12 occupation categories by contact volume without crowding labels.

##### 2. What is/are the insight(s) found from the chart?

Blue-collar (21.5%), management (20.9%) and technician (16.8%) receive the most calls; student and housemaid are contacted least.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Negative-growth risk flagged: contact volume does not track conversion potential (see Chart 6) - the campaign is currently allocating the most calls to some of the lower-converting occupations.

#### Chart - 4 - Account Balance Distribution ( Univariate )

In [ ]:
# Chart - 4 visualization code

fig, ax = plt.subplots(1, 2, figsize=(12,4))
lo, hi = df["balance"].quantile([0.01, 0.99])
sns.histplot(df.loc[df["balance"].between(lo, hi), "balance"], bins=60, color=COL_NO, ax=ax[0])
ax[0].axvline(df["balance"].median(), color=COL_YES, ls="--", label=f"median = {df["balance"].median():,.0f}")
ax[0].set(title="Balance (1st-99th percentile shown)", xlabel="Average yearly balance (EUR)", ylabel="Clients")
ax[0].legend()

sns.histplot(df["balance_slog"], bins=60, color=COL_YES, ax=ax[1])
ax[1].set(title="Signed-log balance: sign(x)*log(1+|x|)", xlabel="signed log balance", ylabel="")
plt.tight_layout()
plt.show()

print("\n")

print(f"Skewness raw = {df["balance"].skew():.1f} | signed-log = {df["balance_slog"].skew():.2f}) | "
      f"negative balances: {df["balance_negative"].mean():.1%} | max = {df["balance"].max():,}")

##### 1. Why did you pick the specific chart?

Balance is extremely right-skewed (skew ≈ 8.4), so a raw histogram (percentile-clipped) is paired with a signed-log transform to show the true shape without a few outliers flattening the plot.

##### 2. What is/are the insight(s) found from the chart?

Median balance is EUR 448 but the mean is EUR 1,362 - pulled up by a max of EUR 102,127. 8.3% of clients carry a negative balance. The signed-log transform reduces skew to -1.6, workable for a linear model.


##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Positive: balance is a usable numeric predictor once transformed/winsorised, and Chart 12 shows it interacts with age on subscription rate.

#### Chart - 5 - Call Volumne by Month ( Univariate )

In [ ]:
# Chart - 5 visualization code

mv= df["month"].value_counts().reindex(MONTH_ORDER)

fig, ax = plt.subplots(figsize=(11, 4))
sns.barplot(x = mv.index, y = mv.values, hue = mv.index, palette = "viridis", legend = False, ax = ax)

for i, v in enumerate(mv.values):
    ax.text(i, v + 150, f"{v/len(df):.0%}", ha = "center", fontsize = 9)

ax.set(title = "Contacts made pre calendar month (all years pooled)", xlabel = "Month", ylabel = "Contacts")
plt.tight_layout()
plt.show()

print(f"May alone = {mv['may']/len(df):.1%} of all contacts; Mar+Sep+Oct+Dec together = {mv[['mar','sep','oct','dec']].sum()/len(df):.1%}")

##### 1. Why did you pick the specific chart?

A simple monthly bar chart is the clearest way to show how uneven the campaign's calendar effort is.

##### 2. What is/are the insight(s) found from the chart?

May alone accounts for 30.4% of all contacts, while March, September, October and December combined make up only 4.4%.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Negative-growth risk: Chart 7 shows the heavily-called months are *not* the highest-converting ones, so a large share of effort is going to the wrong part of the calendar.

#### Chart - 6 - Subscription rate by occupation ( Bivariate : categorical vs target )

In [ ]:
help(rate_table)

In [ ]:
t

In [ ]:
# Chart - 6 visualization code

t = rate_table(df, "job").sort_values("rate")

fig, ax = plt.subplots(figsize = (10,5))
colors = [COL_YES if r > BASE_RATE else COL_NO for r in t["rate"]]
ax.barh(t.index, t["rate"], color = colors)
for i, (r, n) in enumerate(
    zip(t["rate"], t["customers"])
):
    ax.text(r + 0.004, i, f"{r:.1%}  (n={n:,})", va = "center", fontsize = 9)
ax.axvline(BASE_RATE, color = "black", ls = "--", lw = 1)
ax.text(BASE_RATE + 0.002, -0.9, f"overall {BASE_RATE:.1%}", fontsize = 9)
ax.xaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.set_xlim(0, t["rate"].max() * 1.3)
ax.set(title = "Subscription rate by occupation", xlabel = "Subscription rate", ylabel ="")
plt.tight_layout()
plt.show()

##### 1. Why did you pick the specific chart?

A sorted horizontal bar with a reference line at the overall base rate makes over/under-performing occupations immediately visible.

##### 2. What is/are the insight(s) found from the chart?

Students (28.7%) and retirees (22.8%) convert far above the 11.7% base rate; blue-collar (7.3%) and entrepreneurs (8.3%) convert well below it - despite blue-collar receiving the *most* calls (Chart 3).

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Positive: occupation alone is a strong, legitimate pre-call targeting signal and is retained in the model (job has meaningful IV in Section 6.4).

#### Chart - 7 - Volume vs success rate by month ( Bivariate )

In [ ]:
# Chart - 7 visualization code

t = rate_table(df, "month", MONTH_ORDER)
fig, ax = plt.subplots(figsize=(11, 4.4))
ax2 = ax.twinx()
ax.bar(t.index, t["customers"], color = "#C9D1DC", label = "Contacts (left)")
ax2.plot(t.index, t["rate"], color = COL_YES, marker = "o", lw = 2.5, label = "Subscription rate (right)")
for i, r in enumerate(t["rate"]):
  ax2.text(i, r + 0.02, f"{r:.0%}", ha = "center", color = COL_YES, fontsize = 9, fontweight = "bold")
ax2.axhline(BASE_RATE, color = "black", ls = ":", lw = 1)
ax2.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax2.set_ylim(0, 0.65)
ax2.grid(False)
ax2.spines["right"].set_visible(True)
ax.set(title = "Where the calls were made vs. where they converted", xlabel = "Month", ylabel = "Contacts")
ax2.set_ylabel("Subscription rate")
fig.legend(loc = "upper right", bbox_to_anchor = (0.9,0.92))
plt.tight_layout()
plt.show()


##### 1. Why did you pick the specific chart?

A dual-axis chart (bars = volume, line = rate) is the only way to show that volume and conversion tell two different stories on the same time axis.

##### 2. What is/are the insight(s) found from the chart?

March, September, October and December convert at 44-52%, roughly 4x the May rate (6.7%) - yet May receives the most calls by far. This is the same imbalance seen in Chart 5, now tied directly to outcome.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Positive/negative both apply: a strong business opportunity (re-timing campaigns toward the high-converting months) is sitting inside an inefficient allocation - this single chart is arguably the clearest ROI lever in the whole EDA.

#### Chart - 8 - Subscription rate by age group ( Bivariate )

In [ ]:
# Chart - 8 visualization code

t = rate_table(df,"age_group")
fig, ax = plt.subplots(figsize = (9,4.5))
sns.barplot(x = t.index, y = t["rate"], hue = t.index, palette = "flare", legend = False, ax = ax)
for i, (r, n) in enumerate(zip(t["rate"], t["customers"])):
  ax.text(i, r + 0.006, f"{r:.1%}\nn={n:,}", ha = "center", fontsize = 9)
ax.axhline(BASE_RATE, color = "black", ls = "--", lw =1)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.set_ylim(0, t["rate"].max() * 1.25)
ax.set(title = "Subscription rate by age group (U- Shaped)", xlabel = "Age group", ylabel = "Subscription rate")
plt.tight_layout()
plt.show()

##### 1. Why did you pick the specific chart?

A bar chart across ordered age bands shows the shape of the age-outcome relationship directly, which a raw scatterplot on a discrete numeric range would not.

##### 2. What is/are the insight(s) found from the chart?

Subscription rate is U-shaped: highest for under-25 and 65+ clients (well above the 11.7% base rate), lowest for the 35-54 working-age bands.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Positive : age band is a cheap, always-available targeting signal; the U-shape also matches the retiree/student occupation effect in Chart 6, reinforcing rather than contradicting it.

#### Chart - 9

In [ ]:
# Chart - 9 visualization code

##### 1. Why did you pick the specific chart?

Answer Here.

##### 2. What is/are the insight(s) found from the chart?

Answer Here

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Answer Here

#### Chart - 10

In [ ]:
# Chart - 10 visualization code

##### 1. Why did you pick the specific chart?

Answer Here.

##### 2. What is/are the insight(s) found from the chart?

Answer Here

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Answer Here

#### Chart - 11

In [ ]:
# Chart - 11 visualization code

##### 1. Why did you pick the specific chart?

Answer Here.

##### 2. What is/are the insight(s) found from the chart?

Answer Here

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Answer Here

#### Chart - 12

In [ ]:
# Chart - 12 visualization code

##### 1. Why did you pick the specific chart?

Answer Here.

##### 2. What is/are the insight(s) found from the chart?

Answer Here

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Answer Here

#### Chart - 13

In [ ]:
# Chart - 13 visualization code

##### 1. Why did you pick the specific chart?

Answer Here.

##### 2. What is/are the insight(s) found from the chart?

Answer Here

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Answer Here

#### Chart - 14 - Correlation Heatmap

In [ ]:
# Correlation Heatmap visualization code

##### 1. Why did you pick the specific chart?

Answer Here.

##### 2. What is/are the insight(s) found from the chart?

Answer Here

#### Chart - 15 - Pair Plot

In [ ]:
# Pair Plot visualization code

##### 1. Why did you pick the specific chart?

Answer Here.

##### 2. What is/are the insight(s) found from the chart?

Answer Here

## ***5. Hypothesis Testing***

### Based on your chart experiments, define three hypothetical statements from the dataset. In the next three questions, perform hypothesis testing to obtain final conclusion about the statements through your code and statistical testing.

Answer Here.

### Hypothetical Statement - 1

#### 1. State Your research hypothesis as a null hypothesis and alternate hypothesis.

Answer Here.

#### 2. Perform an appropriate statistical test.

In [ ]:
# Perform Statistical Test to obtain P-Value

##### Which statistical test have you done to obtain P-Value?

Answer Here.

##### Why did you choose the specific statistical test?

Answer Here.

### Hypothetical Statement - 2

#### 1. State Your research hypothesis as a null hypothesis and alternate hypothesis.

Answer Here.

#### 2. Perform an appropriate statistical test.

In [ ]:
# Perform Statistical Test to obtain P-Value

##### Which statistical test have you done to obtain P-Value?

Answer Here.

##### Why did you choose the specific statistical test?

Answer Here.

### Hypothetical Statement - 3

#### 1. State Your research hypothesis as a null hypothesis and alternate hypothesis.

Answer Here.

#### 2. Perform an appropriate statistical test.

In [ ]:
# Perform Statistical Test to obtain P-Value

##### Which statistical test have you done to obtain P-Value?

Answer Here.

##### Why did you choose the specific statistical test?

Answer Here.

## ***6. Feature Engineering & Data Pre-processing***

### 1. Handling Missing Values

In [ ]:
# Handling Missing Values & Missing Value Imputation

#### What all missing value imputation techniques have you used and why did you use those techniques?

Answer Here.

### 2. Handling Outliers

In [ ]:
# Handling Outliers & Outlier treatments

##### What all outlier treatment techniques have you used and why did you use those techniques?

Answer Here.

### 3. Categorical Encoding

In [ ]:
# Encode your categorical columns

#### What all categorical encoding techniques have you used & why did you use those techniques?

Answer Here.

### 4. Textual Data Preprocessing
(It's mandatory for textual dataset i.e., NLP, Sentiment Analysis, Text Clustering etc.)

#### 1. Expand Contraction

In [ ]:
# Expand Contraction

#### 2. Lower Casing

In [ ]:
# Lower Casing

#### 3. Removing Punctuations

In [ ]:
# Remove Punctuations

#### 4. Removing URLs & Removing words and digits contain digits.

In [ ]:
# Remove URLs & Remove words and digits contain digits

#### 5. Removing Stopwords & Removing White spaces

In [ ]:
# Remove Stopwords

In [ ]:
# Remove White spaces

#### 6. Rephrase Text

In [ ]:
# Rephrase Text

#### 7. Tokenization

In [ ]:
# Tokenization

#### 8. Text Normalization

In [ ]:
# Normalizing Text (i.e., Stemming, Lemmatization etc.)

##### Which text normalization technique have you used and why?

Answer Here.

#### 9. Part of speech tagging

In [ ]:
# POS Taging

#### 10. Text Vectorization

In [ ]:
# Vectorizing Text

##### Which text vectorization technique have you used and why?

Answer Here.

### 4. Feature Manipulation & Selection

#### 1. Feature Manipulation

In [ ]:
# Manipulate Features to minimize feature correlation and create new features

#### 2. Feature Selection

In [ ]:
# Select your features wisely to avoid overfitting

##### What all feature selection methods have you used  and why?

Answer Here.

##### Which all features you found important and why?

Answer Here.

### 5. Data Transformation

#### Do you think that your data needs to be transformed? If yes, which transformation have you used. Explain Why?

In [ ]:
# Transform Your data

### 6. Data Scaling

In [ ]:
# Scaling your data

##### Which method have you used to scale you data and why?

### 7. Dimesionality Reduction

##### Do you think that dimensionality reduction is needed? Explain Why?

Answer Here.

In [ ]:
# DImensionality Reduction (If needed)

##### Which dimensionality reduction technique have you used and why? (If dimensionality reduction done on dataset.)

Answer Here.

### 8. Data Splitting

In [ ]:
# Split your data to train and test. Choose Splitting ratio wisely.

##### What data splitting ratio have you used and why?

Answer Here.

### 9. Handling Imbalanced Dataset

##### Do you think the dataset is imbalanced? Explain Why.

Answer Here.

In [ ]:
# Handling Imbalanced Dataset (If needed)

##### What technique did you use to handle the imbalance dataset and why? (If needed to be balanced)

Answer Here.

## ***7. ML Model Implementation***

### ML Model - 1

In [ ]:
# ML Model - 1 Implementation

# Fit the Algorithm

# Predict on the model

#### 1. Explain the ML Model used and it's performance using Evaluation metric Score Chart.

In [ ]:
# Visualizing evaluation Metric Score chart

#### 2. Cross- Validation & Hyperparameter Tuning

In [ ]:
# ML Model - 1 Implementation with hyperparameter optimization techniques (i.e., GridSearch CV, RandomSearch CV, Bayesian Optimization etc.)

# Fit the Algorithm

# Predict on the model

##### Which hyperparameter optimization technique have you used and why?

Answer Here.

##### Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

Answer Here.

### ML Model - 2

#### 1. Explain the ML Model used and it's performance using Evaluation metric Score Chart.

In [ ]:
# Visualizing evaluation Metric Score chart

#### 2. Cross- Validation & Hyperparameter Tuning

In [ ]:
# ML Model - 1 Implementation with hyperparameter optimization techniques (i.e., GridSearch CV, RandomSearch CV, Bayesian Optimization etc.)

# Fit the Algorithm

# Predict on the model

##### Which hyperparameter optimization technique have you used and why?

Answer Here.

##### Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

Answer Here.

#### 3. Explain each evaluation metric's indication towards business and the business impact pf the ML model used.

Answer Here.

### ML Model - 3

In [ ]:
# ML Model - 3 Implementation

# Fit the Algorithm

# Predict on the model

#### 1. Explain the ML Model used and it's performance using Evaluation metric Score Chart.

In [ ]:
# Visualizing evaluation Metric Score chart

#### 2. Cross- Validation & Hyperparameter Tuning

In [ ]:
# ML Model - 3 Implementation with hyperparameter optimization techniques (i.e., GridSearch CV, RandomSearch CV, Bayesian Optimization etc.)

# Fit the Algorithm

# Predict on the model

##### Which hyperparameter optimization technique have you used and why?

Answer Here.

##### Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

Answer Here.

### 1. Which Evaluation metrics did you consider for a positive business impact and why?

Answer Here.

### 2. Which ML model did you choose from the above created models as your final prediction model and why?

Answer Here.

### 3. Explain the model which you have used and the feature importance using any model explainability tool?

Answer Here.

## ***8.*** ***Future Work (Optional)***

### 1. Save the best performing ml model in a pickle file or joblib file format for deployment process.


In [ ]:
# Save the File

### 2. Again Load the saved model file and try to predict unseen data for a sanity check.


In [ ]:
# Load the File and predict unseen data.

### ***Congrats! Your model is successfully created and ready for deployment on a live server for a real user interaction !!!***

# **Conclusion**

Write the conclusion here.

### ***Hurrah! You have successfully completed your Machine Learning Capstone Project !!!***